# Binary Conspiracy Detection — RoBERTa-Large + LoRA

This notebook reproduces **train_and_infer_binary.py**:
- **Train/Val**: 90/10 stratified split from `train_rehydrated.jsonl` (Yes/No only)
- **Test**: Full dev — text from `dev_rehydrated.jsonl`, labels from `dev_public.jsonl`
- **Submission**: Inference on `test_rehydrated.jsonl` → `submission_output/submission.jsonl` + `.zip`

**Note:** Run from project root so paths resolve correctly, or set `BASE` in the config cell to your project root.

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
    precision_recall_fscore_support,
)

from transformers import (
    RobertaTokenizerFast,
    RobertaForSequenceClassification,
    RobertaConfig,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("Imports and seeds set.")
print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Helper functions

In [ ]:
def load_and_filter_data(file_path):
    """Load JSONL and filter to binary classification (Yes/No only)."""
    data = []
    with open(file_path, "r") as f:
        for line in f:
            try:
                item = json.loads(line)
                if "conspiracy" in item and item["conspiracy"] in ["Yes", "No"]:
                    data.append({
                        "_id": item.get("_id", ""),
                        "text": item.get("text", ""),
                        "conspiracy": item["conspiracy"],
                    })
            except json.JSONDecodeError:
                print("Skipping invalid JSON line")
    return data


def load_dev_data(file_path):
    """Load dev/test data for inference (id + text only)."""
    data = []
    with open(file_path, "r") as f:
        for i, line in enumerate(f):
            try:
                item = json.loads(line)
                data.append({
                    "unique_sample_id": item.get("_id", f"sample_{i}"),
                    "text": item.get("text", ""),
                })
            except json.JSONDecodeError:
                print(f"Skipping invalid JSON at line {i}")
    return data


def load_test_from_dev(dev_public_path, dev_rehydrated_path):
    """Build test set: labels from dev_public, text from dev_rehydrated (Yes/No only)."""
    with open(dev_rehydrated_path, "r") as f:
        id_to_text = {json.loads(line)["_id"]: json.loads(line).get("text", "") for line in f}
    data = []
    with open(dev_public_path, "r") as f:
        for line in f:
            try:
                item = json.loads(line)
                _id = item.get("_id")
                label = item.get("conspiracy", "")
                if _id is None or label not in ("Yes", "No"):
                    continue
                data.append({"_id": _id, "text": id_to_text.get(_id, ""), "conspiracy": label})
            except json.JSONDecodeError:
                pass
    return data


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=-1)
    accuracy = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average="macro")
    f1_weighted = f1_score(labels, predictions, average="weighted")
    return {"accuracy": accuracy, "f1_macro": f1_macro, "f1_weighted": f1_weighted}


def load_dev_public_gold(file_path):
    """Load dev_public.jsonl: _id -> conspiracy."""
    gold = {}
    with open(file_path, "r") as f:
        for line in f:
            try:
                item = json.loads(line)
                _id = item.get("_id")
                if _id is not None:
                    gold[_id] = item.get("conspiracy", "")
            except json.JSONDecodeError:
                pass
    return gold


def evaluate_vs_dev_public(unique_ids, predicted_labels, gold_path, output_dir):
    """Compare predictions to dev_public ground truth (Yes/No only); save scores.json."""
    gold = load_dev_public_gold(gold_path)
    y_true, y_pred = [], []
    n_cant_tell = 0
    for i, _id in enumerate(unique_ids):
        g = gold.get(_id)
        if g is None:
            continue
        if g not in ("Yes", "No"):
            n_cant_tell += 1
            continue
        y_true.append(g)
        y_pred.append(predicted_labels[i])
    if n_cant_tell:
        print(f"  (Excluded {n_cant_tell} dev_public samples with 'Can't tell' from metrics)")
    if not y_true:
        print("  No Yes/No ground-truth samples to evaluate.")
        return
    acc = accuracy_score(y_true, y_pred)
    f1w = f1_score(y_true, y_pred, labels=["No", "Yes"], average="weighted", zero_division=0)
    f1m = f1_score(y_true, y_pred, labels=["No", "Yes"], average="macro", zero_division=0)
    prec, rec, f1_per, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=["No", "Yes"], average=None, zero_division=0
    )
    print(f"\n  Matched {len(y_true)} samples (Yes/No only).")
    print(f"  Accuracy: {acc:.4f}  F1 (weighted): {f1w:.4f}  F1 (macro): {f1m:.4f}")
    print("\n  Classification report vs dev_public:")
    print(classification_report(y_true, y_pred, labels=["No", "Yes"]))
    cm = confusion_matrix(y_true, y_pred, labels=["No", "Yes"])
    print("  Confusion matrix (rows=true, cols=pred):")
    print(f"    {cm}")
    scores = {
        "accuracy": float(acc),
        "f1_weighted": float(f1w),
        "f1_macro": float(f1m),
        "f1_No": float(f1_per[0]),
        "f1_Yes": float(f1_per[1]),
        "n_eval": len(y_true),
        "n_cant_tell_excluded": n_cant_tell,
    }
    out_path = Path(output_dir) / "scores.json"
    with open(out_path, "w") as f:
        json.dump(scores, f, indent=2)
    print(f"\n  Saved metrics to {out_path}")

## Configuration

In [ ]:
# EDA-Rehydrated root: when running from EDA-Rehydrated/notebooks, use ..
BASE = Path("..").resolve()

MODEL_NAME = "roberta-large"
MAX_LENGTH = 512
BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 6e-5
NUM_EPOCHS = 10
WEIGHT_DECAY = 0
WARMUP_RATIO = 0
DROPOUT_RATE = 0

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
TARGET_MODULES = ["query", "value", "key", "dense"]

OUTPUT_DIR = BASE / "models" / "roberta-large-binary-conspiracy-lora"
SPLITS_DIR = BASE / "data_splits"
SPLITS_DIR.mkdir(exist_ok=True)

label_to_id = {"No": 0, "Yes": 1}
id_to_label = {0: "No", 1: "Yes"}
num_labels = 2

print(f"BASE: {BASE}")
print(f"Model: {MODEL_NAME}")
print(f"Output dir: {OUTPUT_DIR}")

## Load data

In [ ]:
train_file = BASE / "data" / "train_rehydrated.jsonl"
train_data = load_and_filter_data(train_file)
df = pd.DataFrame(train_data)

print(f"Loaded {len(df)} samples (binary Yes/No only)")
print("\nClass distribution:")
print(df["conspiracy"].value_counts())

In [ ]:
train_ids_file = SPLITS_DIR / "binary_train_ids.txt"
val_ids_file = SPLITS_DIR / "binary_val_ids.txt"
dev_public_path = BASE / "data" / "dev_public.jsonl"
dev_rehydrated_path = BASE / "data" / "dev_rehydrated.jsonl"

if train_ids_file.exists() and val_ids_file.exists():
    print("Loading existing 90/10 train/val splits...")
    with open(train_ids_file, "r") as f:
        train_ids = set(line.strip() for line in f)
    with open(val_ids_file, "r") as f:
        val_ids = set(line.strip() for line in f)
    train_df = df[df["_id"].isin(train_ids)].copy()
    val_df = df[df["_id"].isin(val_ids)].copy()
else:
    print("Creating 90/10 train/val splits...")
    train_df, val_df = train_test_split(
        df, test_size=0.1, stratify=df["conspiracy"], random_state=42
    )
    with open(train_ids_file, "w") as f:
        f.write("\n".join(train_df["_id"].values))
    with open(val_ids_file, "w") as f:
        f.write("\n".join(val_df["_id"].values))
    print(f"Saved split IDs to {SPLITS_DIR}")

test_data = load_test_from_dev(dev_public_path, dev_rehydrated_path)
test_df = pd.DataFrame(test_data)

print(f"\nTrain: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")

## Prepare datasets (tokenize)

In [ ]:
tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=MAX_LENGTH)

train_texts = train_df["text"].tolist()
train_labels = [label_to_id[label] for label in train_df["conspiracy"].tolist()]
val_texts = val_df["text"].tolist()
val_labels = [label_to_id[label] for label in val_df["conspiracy"].tolist()]
test_texts = test_df["text"].tolist()
test_labels = [label_to_id[label] for label in test_df["conspiracy"].tolist()]

train_dataset = Dataset.from_dict({"text": train_texts, "labels": train_labels})
val_dataset = Dataset.from_dict({"text": val_texts, "labels": val_labels})
test_dataset = Dataset.from_dict({"text": test_texts, "labels": test_labels})

train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
val_dataset = val_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
test_dataset = test_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

print("Datasets tokenized and ready.")

## Initialize model (RoBERTa-Large + LoRA)

In [ ]:
config = RobertaConfig.from_pretrained(MODEL_NAME)
config.hidden_dropout_prob = DROPOUT_RATE
config.attention_probs_dropout_prob = DROPOUT_RATE
config.num_labels = num_labels
config.id2label = id_to_label
config.label2id = label_to_id

model = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, config=config)

if torch.cuda.is_available():
    model = model.to("cuda")
    print("Model moved to GPU")

if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()
    print("Gradient checkpointing enabled")

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
)
model = get_peft_model(model, lora_config)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"LoRA applied. Trainable: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.2f}%)")

## Training

In [ ]:
use_fp16 = torch.cuda.is_available()
use_bf16 = False
if torch.cuda.is_available():
    try:
        _ = torch.tensor([1.0], dtype=torch.bfloat16)
        use_bf16 = True
        use_fp16 = False
    except Exception:
        pass

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    logging_dir=str(BASE / "logs"),
    logging_steps=50,
    report_to="none",
    seed=42,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    fp16=use_fp16,
    bf16=use_bf16,
    dataloader_num_workers=0,
    save_total_limit=2,
    gradient_checkpointing=True,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True)
early_stopping = EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.001)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[early_stopping],
)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Starting training...")
train_result = trainer.train()
print(f"\nTraining complete. Loss: {train_result.training_loss:.4f}")

## Validation evaluation

In [ ]:
val_results = trainer.evaluate(val_dataset)
print(f"Accuracy: {val_results['eval_accuracy']:.4f}")
print(f"F1 (macro): {val_results['eval_f1_macro']:.4f}")
print(f"F1 (weighted): {val_results['eval_f1_weighted']:.4f}")

## Test evaluation (dev set)

In [ ]:
test_results = trainer.evaluate(test_dataset)
print(f"Accuracy: {test_results['eval_accuracy']:.4f}")
print(f"F1 (macro): {test_results['eval_f1_macro']:.4f}")
print(f"F1 (weighted): {test_results['eval_f1_weighted']:.4f}")

predictions = trainer.predict(test_dataset)
predicted_classes = np.argmax(predictions.predictions, axis=-1)
y_true_labels = [id_to_label[lbl] for lbl in test_dataset["labels"]]
y_pred_labels = [id_to_label[int(c)] for c in predicted_classes]

print("\nClassification report (test = dev):")
print(classification_report(y_true_labels, y_pred_labels, labels=["No", "Yes"]))

## Generate submission (test_rehydrated.jsonl → submission_output/)

In [ ]:
SUBMISSION_INPUT_FILE = BASE / "data" / "test_rehydrated.jsonl"
SUBMISSION_OUTPUT_DIR = BASE / "submission_output"
SUBMISSION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_FILE = SUBMISSION_OUTPUT_DIR / "submission.jsonl"
SUBMISSION_ZIP = SUBMISSION_OUTPUT_DIR / "submission.zip"

submission_data = load_dev_data(SUBMISSION_INPUT_FILE)
submission_dataset = Dataset.from_list(submission_data)
unique_ids = submission_dataset["unique_sample_id"]

submission_dataset_tokenized = submission_dataset.map(
    lambda examples: tokenizer(examples["text"], truncation=True, max_length=MAX_LENGTH),
    batched=True,
)
submission_dataset_tokenized = submission_dataset_tokenized.remove_columns(["unique_sample_id", "text"])

print(f"Loaded {len(submission_dataset)} samples from {SUBMISSION_INPUT_FILE.name}")
print("Generating predictions...")
predictions_output = trainer.predict(submission_dataset_tokenized)
predicted_class_ids = np.argmax(predictions_output.predictions, axis=-1)
predicted_labels = [id_to_label[int(i)] for i in predicted_class_ids]

print("Prediction distribution:")
print(pd.Series(predicted_labels).value_counts())

with open(SUBMISSION_FILE, "w") as f:
    for i, label in enumerate(predicted_labels):
        f.write(json.dumps({"_id": unique_ids[i], "conspiracy": label}) + "\n")

with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(SUBMISSION_FILE, arcname="submission.jsonl")

print(f"\nSaved {SUBMISSION_FILE}")
print(f"Created {SUBMISSION_ZIP}")

## Evaluation vs dev_public.jsonl (ground truth)

In [ ]:
DEV_PUBLIC_GT = BASE / "data" / "dev_public.jsonl"
if DEV_PUBLIC_GT.exists():
    evaluate_vs_dev_public(
        list(unique_ids), predicted_labels, DEV_PUBLIC_GT, SUBMISSION_OUTPUT_DIR
    )
else:
    print(f"Ground truth not found: {DEV_PUBLIC_GT}")

## Summary

- **Output dir:** `submission_output/`  
- **Submission zip:** `submission_output/submission.zip`  
- **Scores:** `submission_output/scores.json` (vs dev_public)  
- **Next:** Upload `submission.zip` to Codalab (target: > 0.76 weighted F1).